# AirLLM vs Ollama vs HuggingFace - Benchmark Results Analysis

This notebook loads saved benchmark results, displays a summary table, and generates five comparison charts (including a parameter-sensitivity line chart).

## 1. Imports and Setup

In [1]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.use("Agg")  # headless backend - swap to 'inline' in interactive Jupyter

# Make the src package importable when running from notebooks/
sys.path.insert(0, str(Path('..') / 'src'))

from airllm_benchmark.visualization.chart_service import ChartService
from airllm_benchmark.visualization.notebook_data import NotebookDataPreparer

RESULTS_DIR = Path('..') / 'results'
ASSETS_DIR  = Path('..') / 'assets'

preparer = NotebookDataPreparer(results_dir=RESULTS_DIR)
charts   = ChartService(config={'assets_dir': str(ASSETS_DIR)})

print('Setup complete.')

Setup complete.


## 2. Load Benchmark Results

In [2]:
results = preparer.load_results()
print(f'Loaded {len(results)} result(s) from {RESULTS_DIR}')
for r in results:
    status = 'OK' if r.is_success else f'FAILED: {r.error}'
    print(f'  {r.method:<14} {r.model_id:<40} {status}')

Loaded 36 result(s) from ..\results
  ollama         llama3.2:3b                              OK
  ollama         llama3.2:3b                              OK
  hf_baseline    TinyLlama/TinyLlama-1.1B-Chat-v1.0       OK
  hf_baseline    TinyLlama/TinyLlama-1.1B-Chat-v1.0       OK
  hf_baseline    TinyLlama/TinyLlama-1.1B-Chat-v1.0       OK
  airllm         mistralai/Mistral-7B-v0.1                OK
  airllm         mistralai/Mistral-7B-v0.1                OK
  hf_baseline    TinyLlama/TinyLlama-1.1B-Chat-v1.0       FAILED: All 4 attempts for 'huggingface' failed
  ollama         llama3.2:3b                              OK
  airllm         mistralai/Mistral-7B-v0.1                OK
  hf_baseline    TinyLlama/TinyLlama-1.1B-Chat-v1.0       OK
  ollama         llama3.2:3b                              OK
  airllm         mistralai/Mistral-7B-v0.1                FAILED: All 3 attempts for 'airllm' failed
  hf_baseline    TinyLlama/TinyLlama-1.1B-Chat-v1.0       OK
  ollama         llama3.2

## 3. Summary Table

In [3]:
df = preparer.summary_dataframe()

if df.empty:
    print('No results found. Run airllm-benchmark first, then re-run this notebook.')
else:
    display_cols = ['method', 'model_id', 'latency_s', 'ram_peak_mb', 'vram_peak_mb',
                    'tokens_per_second', 'tokens_generated', 'cost_estimate', 'is_success']
    pd.set_option('display.max_colwidth', 50)
    display(df[display_cols].style.format({
        'latency_s':       '{:.2f}',
        'ram_peak_mb':     '{:.0f}',
        'vram_peak_mb':    '{:.0f}',
        'tokens_per_second': '{:.1f}',
    }))

,method,model_id,latency_s,ram_peak_mb,vram_peak_mb,tokens_per_second,tokens_generated,cost_estimate,is_success
0,ollama,llama3.2:3b,9.61,33,0,113.5,20,~0.00053 kWh @ 200W TDP,True
1,ollama,llama3.2:3b,2.68,33,0,120.4,20,~0.00015 kWh @ 200W TDP,True
2,hf_baseline,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4.58,4559,0,0.2,1,~0.00025 kWh @ 200W TDP,True
3,hf_baseline,TinyLlama/TinyLlama-1.1B-Chat-v1.0,3.88,6571,0,0.3,1,~0.00022 kWh @ 200W TDP,True
4,hf_baseline,TinyLlama/TinyLlama-1.1B-Chat-v1.0,9.15,6618,0,2.2,20,~0.00051 kWh @ 200W TDP,True
5,airllm,mistralai/Mistral-7B-v0.1,394.42,1635,0,0.1,20,~0.00712 kWh @ 65W TDP,True
6,airllm,mistralai/Mistral-7B-v0.1,370.97,1637,0,0.1,20,~0.00670 kWh @ 65W TDP,True
7,hf_baseline,TinyLlama/TinyLlama-1.1B-Chat-v1.0,0.00,0,0,0.0,0,,False
8,ollama,llama3.2:3b,12.97,197,0,126.8,20,~0.00072 kWh @ 200W TDP,True
9,airllm,mistralai/Mistral-7B-v0.1,392.59,4011,0,0.1,20,~0.00709 kWh @ 65W TDP,True


## 4. Latency Bar Chart

In [4]:
if results:
    path = charts.plot_latency_bar(results)
    print(f'Saved -> {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

Saved -> ..\assets\latency_bar.png


C:\Users\atrab\AppData\Local\Temp\ipykernel_5952\1543257111.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Memory Comparison Chart (RAM vs VRAM)

In [5]:
if results:
    path = charts.plot_memory_grouped_bar(results)
    print(f'Saved -> {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

Saved -> ..\assets\memory_grouped_bar.png


C:\Users\atrab\AppData\Local\Temp\ipykernel_5952\1966767096.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Throughput Bar Chart (tokens/s)

In [6]:
if results:
    path = charts.plot_throughput_bar(results)
    print(f'Saved -> {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

Saved -> ..\assets\throughput_bar.png


C:\Users\atrab\AppData\Local\Temp\ipykernel_5952\1626566867.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Trade-off Scatter Plot (Latency vs RAM)

In [7]:
if results:
    path = charts.plot_trade_off_scatter(results)
    print(f'Saved -> {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

Saved -> ..\assets\trade_off_scatter.png


C:\Users\atrab\AppData\Local\Temp\ipykernel_5952\3622593740.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Parameter Sensitivity: Latency vs. max_tokens (Ollama)

In [8]:
token_counts = [5, 10, 20, 40, 80]
latencies_s = [2.622, 2.773, 2.823, 3.024, 2.976]  # measured, warm llama3.2:3b

path = charts.plot_latency_vs_tokens_line(token_counts, latencies_s, 'ollama')
print(f'Saved -> {path}')
img = plt.imread(str(path))
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Latency is essentially flat across a 16x range in max_tokens -- for this small,')
print('already-warm model, fixed request overhead dominates over per-token generation cost.')


C:\Users\atrab\AppData\Local\Temp\ipykernel_5952\2414404059.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved -> ..\assets\latency_vs_tokens_line.png
Latency is essentially flat across a 16x range in max_tokens -- for this small,
already-warm model, fixed request overhead dominates over per-token generation cost.


## 9. Cost Analysis and Conclusions

In [9]:
print(preparer.hardware_profile_summary())
print()

if not df.empty:
    successful = df[df['is_success']]
    if not successful.empty:
        fastest = successful.loc[successful['latency_s'].idxmin(), 'method']
        most_efficient = successful.loc[successful['ram_peak_mb'].idxmin(), 'method']
        highest_tps = successful.loc[successful['tokens_per_second'].idxmax(), 'method']
        print(f'Fastest method       : {fastest}')
        print(f'Most RAM-efficient   : {most_efficient}')
        print(f'Highest throughput   : {highest_tps}')
        print()
        print('Cost estimates (energy at TDP):')
        for _, row in successful.iterrows():
            print(f'  {row["method"]:<14}: {row["cost_estimate"]}')
        print()
        print('Conclusion:')
        print('  AirLLM enables running models that exceed available VRAM via CPU layer paging,')
        print('  at the cost of significantly higher latency compared to Ollama or standard HF.')
        print('  Use Ollama for fast inference on small models; AirLLM for large models on low-VRAM hardware.')
    else:
        print('No successful results to analyse.')
else:
    print('Run airllm-benchmark to generate results, then re-run this notebook.')

Hardware: Intel i7-4790K | RTX 3060 Ti 8 GB VRAM | 32 GB DDR3
AirLLM target: Mistral-7B-v0.1 (14 GB fp16) — exceeds VRAM, runs via CPU layer paging

Fastest method       : ollama
Most RAM-efficient   : ollama
Highest throughput   : ollama

Cost estimates (energy at TDP):
  ollama        : ~0.00053 kWh @ 200W TDP
  ollama        : ~0.00015 kWh @ 200W TDP
  hf_baseline   : ~0.00025 kWh @ 200W TDP
  hf_baseline   : ~0.00022 kWh @ 200W TDP
  hf_baseline   : ~0.00051 kWh @ 200W TDP
  airllm        : ~0.00712 kWh @ 65W TDP
  airllm        : ~0.00670 kWh @ 65W TDP
  ollama        : ~0.00072 kWh @ 200W TDP
  airllm        : ~0.00709 kWh @ 65W TDP
  hf_baseline   : ~0.00078 kWh @ 200W TDP
  ollama        : ~0.00047 kWh @ 200W TDP
  hf_baseline   : ~0.00093 kWh @ 200W TDP
  ollama        : ~0.00063 kWh @ 200W TDP
  hf_baseline   : ~0.00074 kWh @ 200W TDP
  ollama        : ~0.00018 kWh @ 200W TDP
  airllm        : ~0.00802 kWh @ 65W TDP
  hf_baseline   : ~0.00055 kWh @ 200W TDP
  ollama        : 